# **이미지 다운로드**

In [ ]:
# input : 정면 사진 (Final_DATA -> front에 들어가셔서 정면 이미지 내 드라이브에 다운 -> 다운 받고 저장한 경로)
# 정답지 : final_GT.json 파일 경로 (457쌍)

# --> train 시도 section 가시면 경로 입력하는 부분이 있어서 꼭 "내 드라이브"에 다운로드 후 경로를 입력해주셔야 합니다 !

In [ ]:
# 기본 token HPE와 다른 점
# 1. MLP Head

#    (1) relu + sigmoid 사용
#        - 코드 : nn.Sigmoid()

#   (2) fm_token + ori_token 이용 -> 파라미터 수 증가 (9 -> 205)
#        - 코드 : total_tokens = num_ori_tokens + fm_tokens

#      --> Linear Projection을 이용해 차원을 축소 (128 -> 16)
#          - 코드  : self.proj_dim = 16
#                    self.projection = nn.Linear(dim, self.proj_dim)

#    (3) 추출 6개 -> 1개
#        - 코드 : nn.Linear(hidden_dim, 1)

# 2. 회전행렬 R 사용 X

# 3. 파라미터 수 약 21배 증가

# **이미지 불러오기**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# timm 패키지 버전에 대해서 처리!--> 수정
!pip uninstall timm
!pip install timm==0.6.13

Found existing installation: timm 1.0.25
Uninstalling timm-1.0.25:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/timm-1.0.25.dist-info/*
    /usr/local/lib/python3.12/dist-packages/timm/*
Proceed (Y/n)? ㅛ
Your response ('ㅛ') was not one of the expected responses: y, n, 
Proceed (Y/n)? Y
  Successfully uninstalled timm-1.0.25
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 17.6 MB/s eta 0:00:00


# **utils.py**

In [ ]:
# utils.py라는 파일을 생성하고 함수나 클래스를 정의
with open('utils.py', 'w') as f:
    f.write("""
import numpy as np
import torch
import scipy.io as sio
import cv2
from math import cos, sin

def plot_pose_cube(img, yaw, pitch, roll, tdx=None, tdy=None, size=150.):
    # Input is a cv2 image
    # pose_params: (pitch, yaw, roll, tdx, tdy)
    # Where (tdx, tdy) is the translation of the face.
    # For pose we have [pitch yaw roll tdx tdy tdz scale_factor]

    p = pitch * np.pi / 180
    y = -(yaw * np.pi / 180)
    r = roll * np.pi / 180
    if tdx != None and tdy != None:
        face_x = tdx - 0.50 * size
        face_y = tdy - 0.50 * size

    else:
        height, width = img.shape[:2]
        face_x = width / 2 - 0.5 * size
        face_y = height / 2 - 0.5 * size

    x1 = size * (cos(y) * cos(r)) + face_x
    y1 = size * (cos(p) * sin(r) + cos(r) * sin(p) * sin(y)) + face_y
    x2 = size * (-cos(y) * sin(r)) + face_x
    y2 = size * (cos(p) * cos(r) - sin(p) * sin(y) * sin(r)) + face_y
    x3 = size * (sin(y)) + face_x
    y3 = size * (-cos(y) * sin(p)) + face_y


    # Draw base in red
    cv2.line(img, (int(face_x), int(face_y)), (int(x1),int(y1)),(0,0,255),3)
    cv2.line(img, (int(face_x), int(face_y)), (int(x2),int(y2)),(0,0,255),3)
    cv2.line(img, (int(x2), int(y2)), (int(x2+x1-face_x),int(y2+y1-face_y)),(0,0,255),3)
    cv2.line(img, (int(x1), int(y1)), (int(x1+x2-face_x),int(y1+y2-face_y)),(0,0,255),3)
    # Draw pillars in blue
    cv2.line(img, (int(face_x), int(face_y)), (int(x3),int(y3)),(255,0,0),2)
    cv2.line(img, (int(x1), int(y1)), (int(x1+x3-face_x),int(y1+y3-face_y)),(255,0,0),2)
    cv2.line(img, (int(x2), int(y2)), (int(x2+x3-face_x),int(y2+y3-face_y)),(255,0,0),2)
    cv2.line(img, (int(x2+x1-face_x),int(y2+y1-face_y)), (int(x3+x1+x2-2*face_x),int(y3+y2+y1-2*face_y)),(255,0,0),2)
    # Draw top in green
    cv2.line(img, (int(x3+x1-face_x),int(y3+y1-face_y)), (int(x3+x1+x2-2*face_x),int(y3+y2+y1-2*face_y)),(0,255,0),2)
    cv2.line(img, (int(x2+x3-face_x),int(y2+y3-face_y)), (int(x3+x1+x2-2*face_x),int(y3+y2+y1-2*face_y)),(0,255,0),2)
    cv2.line(img, (int(x3), int(y3)), (int(x3+x1-face_x),int(y3+y1-face_y)),(0,255,0),2)
    cv2.line(img, (int(x3), int(y3)), (int(x3+x2-face_x),int(y3+y2-face_y)),(0,255,0),2)

    return img

def draw_axis(img, yaw, pitch, roll, tdx=None, tdy=None, size = 100):

    pitch = pitch * np.pi / 180
    yaw = -(yaw * np.pi / 180)
    roll = roll * np.pi / 180

    if tdx != None and tdy != None:
        tdx = tdx
        tdy = tdy
    else:
        height, width = img.shape[:2]
        tdx = width / 2
        tdy = height / 2

    # X-Axis pointing to right. drawn in red
    x1 = size * (cos(yaw) * cos(roll)) + tdx
    y1 = size * (cos(pitch) * sin(roll) + cos(roll) * sin(pitch) * sin(yaw)) + tdy

    # Y-Axis | drawn in green
    #        v
    x2 = size * (-cos(yaw) * sin(roll)) + tdx
    y2 = size * (cos(pitch) * cos(roll) - sin(pitch) * sin(yaw) * sin(roll)) + tdy

    # Z-Axis (out of the screen) drawn in blue
    x3 = size * (sin(yaw)) + tdx
    y3 = size * (-cos(yaw) * sin(pitch)) + tdy

    cv2.line(img, (int(tdx), int(tdy)), (int(x1),int(y1)),(0,0,255),4)
    cv2.line(img, (int(tdx), int(tdy)), (int(x2),int(y2)),(0,255,0),4)
    cv2.line(img, (int(tdx), int(tdy)), (int(x3),int(y3)),(255,0,0),4)

    return img


def get_pose_params_from_mat(mat_path):
    # This functions gets the pose parameters from the .mat
    # Annotations that come with the Pose_300W_LP dataset.
    mat = sio.loadmat(mat_path)
    # [pitch yaw roll tdx tdy tdz scale_factor]
    pre_pose_params = mat['Pose_Para'][0]
    # Get [pitch, yaw, roll, tdx, tdy]
    pose_params = pre_pose_params[:5]
    return pose_params

def get_ypr_from_mat(mat_path):
    # Get yaw, pitch, roll from .mat annotation.
    # They are in radians
    mat = sio.loadmat(mat_path)
    # [pitch yaw roll tdx tdy tdz scale_factor]
    pre_pose_params = mat['Pose_Para'][0]
    # Get [pitch, yaw, roll]
    pose_params = pre_pose_params[:3]
    return pose_params

def get_pt2d_from_mat(mat_path):
    # Get 2D landmarks
    mat = sio.loadmat(mat_path)
    pt2d = mat['pt2d']
    return pt2d

# batch*n
def normalize_vector( v, use_gpu=True):
    batch=v.shape[0]
    v_mag = torch.sqrt(v.pow(2).sum(1))# batch
    if use_gpu:
        v_mag = torch.max(v_mag, torch.autograd.Variable(torch.FloatTensor([1e-8]).cuda()))
    else:
        v_mag = torch.max(v_mag, torch.autograd.Variable(torch.FloatTensor([1e-8])))
    v_mag = v_mag.view(batch,1).expand(batch,v.shape[1])
    v = v/v_mag
    return v

# u, v batch*n
def cross_product( u, v):
    batch = u.shape[0]
    #print (u.shape)
    #print (v.shape)
    i = u[:,1]*v[:,2] - u[:,2]*v[:,1]
    j = u[:,2]*v[:,0] - u[:,0]*v[:,2]
    k = u[:,0]*v[:,1] - u[:,1]*v[:,0]

    out = torch.cat((i.view(batch,1), j.view(batch,1), k.view(batch,1)),1)#batch*3

    return out


#poses batch*6
#poses
def compute_rotation_matrix_from_ortho6d(poses, use_gpu=True):
    x_raw = poses[:,0:3]#batch*3
    y_raw = poses[:,3:6]#batch*3

    x = normalize_vector(x_raw, use_gpu) #batch*3
    z = cross_product(x,y_raw) #batch*3
    z = normalize_vector(z, use_gpu)#batch*3
    y = cross_product(z,x)#batch*3

    x = x.view(-1,3,1)
    y = y.view(-1,3,1)
    z = z.view(-1,3,1)
    matrix = torch.cat((x,y,z), 2) #batch*3*3
    return matrix


#input batch*4*4 or batch*3*3
#output torch batch*3 x, y, z in radiant
#the rotation is in the sequence of x,y,z
def compute_euler_angles_from_rotation_matrices(rotation_matrices, use_gpu=True):
    batch=rotation_matrices.shape[0]
    R=rotation_matrices
    sy = torch.sqrt(R[:,0,0]*R[:,0,0]+R[:,1,0]*R[:,1,0])
    singular= sy<1e-6
    singular=singular.float()

    x=torch.atan2(R[:,2,1], R[:,2,2])
    y=torch.atan2(-R[:,2,0], sy)
    z=torch.atan2(R[:,1,0],R[:,0,0])

    xs=torch.atan2(-R[:,1,2], R[:,1,1])
    ys=torch.atan2(-R[:,2,0], sy)
    zs=R[:,1,0]*0

    if use_gpu:
        out_euler=torch.autograd.Variable(torch.zeros(batch,3).cuda())
    else:
        out_euler=torch.autograd.Variable(torch.zeros(batch,3))
    out_euler[:,0]=x*(1-singular)+xs*singular
    out_euler[:,1]=y*(1-singular)+ys*singular
    out_euler[:,2]=z*(1-singular)+zs*singular

    return out_euler


def get_R(x,y,z):

    # x pitch
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(x), -np.sin(x)],
                   [0, np.sin(x), np.cos(x)]])
    # y yaw
    Ry = np.array([[np.cos(y), 0, np.sin(y)],
                   [0, 1, 0],
                   [-np.sin(y), 0, np.cos(y)]])
    # z roll
    Rz = np.array([[np.cos(z), -np.sin(z), 0],
                   [np.sin(z), np.cos(z), 0],
                   [0, 0, 1]])

    R = Rz.dot(Ry.dot(Rx))
    return R
""")

# **ViT_model.py**

In [ ]:

with open('ViT_model.py', 'w') as f:
    f.write("""
from functools import partial
from collections import OrderedDict

import torch
import torch.nn as nn



def drop_path(x, drop_prob: float = 0., training: bool = False):

    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)  # work with diff dim tensors, not just 2D ConvNets
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()  # binarize
    output = x.div(keep_prob) * random_tensor
    return output


class DropPath(nn.Module):

    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


class PatchEmbed(nn.Module):

    def __init__(self, img_size=224, patch_size=16, in_c=3, embed_dim=768, norm_layer=None):
        super().__init__()
        img_size = (img_size, img_size)
        patch_size = (patch_size, patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1]) # 14*14
        self.num_patches = self.grid_size[0] * self.grid_size[1] # 14*14
                                                      # 16                  16
        self.proj = nn.Conv2d(in_c, embed_dim, kernel_size=patch_size, stride=patch_size)
        #                      3         768                    16                16
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        # flatten: [B, C, H, W] -> [B, C, HW]
        # transpose: [B, C, HW] -> [B, HW, C]
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

    def relprop(self, cam, **kwargs):
        cam = cam.transpose(1,2)
        cam = cam.reshape(cam.shape[0], cam.shape[1],
                     (self.img_size[0] // self.patch_size[0]), (self.img_size[1] // self.patch_size[1]))
        return self.proj.relprop(cam, **kwargs)



class Attention(nn.Module):
    def __init__(self,
                 dim,   # input token dim
                 num_heads=8,
                 qkv_bias=False,
                 qk_scale=None,
                 attn_drop_ratio=0.,
                 proj_drop_ratio=0.):
        super(Attention, self).__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop_ratio)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop_ratio)

        self.attn_cam = None
        self.attn = None
        self.v = None
        self.v_cam = None
        self.attn_gradients = None

    def get_attn(self):
        return self.attn

    def save_attn(self, attn):
        self.attn = attn

    def save_attn_cam(self, cam):
        self.attn_cam = cam

    def get_attn_cam(self):
        return self.attn_cam

    def get_v(self):
        return self.v

    def save_v(self, v):
        self.v = v

    def save_v_cam(self, cam):
        self.v_cam = cam

    def get_v_cam(self):
        return self.v_cam

    def save_attn_gradients(self, attn_gradients):
        self.attn_gradients = attn_gradients

    def get_attn_gradients(self):
        return self.attn_gradients

    def forward(self, x):
        # [batch_size, num_patches + 1, total_embed_dim]
        B, N, C = x.shape

        # qkv(): -> [batch_size, num_patches + 1, 3 * total_embed_dim]
        #                           num_patches + class token
        # reshape: -> [batch_size, num_patches + 1, 3, num_heads, embed_dim_per_head]
        # permute: -> [3, batch_size, num_heads, num_patches + 1, embed_dim_per_head]
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        # [batch_size, num_heads, num_patches + 1, embed_dim_per_head]
        q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)

        # transpose: -> [batch_size, num_heads, embed_dim_per_head, num_patches + 1]
        # @: multiply -> [batch_size, num_heads, num_patches + 1, num_patches + 1]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # @: multiply -> [batch_size, num_heads, num_patches + 1, embed_dim_per_head]
        # transpose: -> [batch_size, num_patches + 1, num_heads, embed_dim_per_head]
        # reshape: -> [batch_size, num_patches + 1, total_embed_dim]
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    def relprop(self, cam, **kwargs):
        cam = self.proj_drop.relprop(cam, **kwargs)
        cam = self.proj.relprop(cam, **kwargs)
        cam = rearrange(cam, 'b n (h d) -> b h n d', h=self.num_heads)

        # attn = A*V
        (cam1, cam_v)= self.matmul2.relprop(cam, **kwargs)
        cam1 /= 2
        cam_v /= 2

        self.save_v_cam(cam_v)
        self.save_attn_cam(cam1)

        cam1 = self.attn_drop.relprop(cam1, **kwargs)
        cam1 = self.softmax.relprop(cam1, **kwargs)

        # A = Q*K^T
        (cam_q, cam_k) = self.matmul1.relprop(cam1, **kwargs)
        cam_q /= 2
        cam_k /= 2

        cam_qkv = rearrange([cam_q, cam_k, cam_v], 'qkv b h n d -> b n (qkv h d)', qkv=3, h=self.num_heads)

        return self.qkv.relprop(cam_qkv, **kwargs)


class Mlp(nn.Module):

    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

    def relprop(self, cam, **kwargs):
        cam = self.drop.relprop(cam, **kwargs)
        cam = self.fc2.relprop(cam, **kwargs)
        cam = self.act.relprop(cam, **kwargs)
        cam = self.fc1.relprop(cam, **kwargs)
        return cam


class Block(nn.Module):
    def __init__(self,
                 dim,
                 num_heads,
                 mlp_ratio=4.,
                 qkv_bias=False,
                 qk_scale=None,
                 drop_ratio=0.,
                 attn_drop_ratio=0.,
                 drop_path_ratio=0.,
                 act_layer=nn.GELU,
                 norm_layer=nn.LayerNorm):
        super(Block, self).__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
                              attn_drop_ratio=attn_drop_ratio, proj_drop_ratio=drop_ratio)
        # NOTE: drop path for stochastic depth, we shall see if this is better than dropout here
        self.drop_path = DropPath(drop_path_ratio) if drop_path_ratio > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop_ratio)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

    def relprop(self, cam, **kwargs):
        (cam1, cam2) = self.add2.relprop(cam, **kwargs)
        cam2 = self.mlp.relprop(cam2, **kwargs)
        cam2 = self.norm2.relprop(cam2, **kwargs)
        cam = self.clone2.relprop((cam1, cam2), **kwargs)

        (cam1, cam2) = self.add1.relprop(cam, **kwargs)
        cam2 = self.attn.relprop(cam2, **kwargs)
        cam2 = self.norm1.relprop(cam2, **kwargs)
        cam = self.clone1.relprop((cam1, cam2), **kwargs)
        return cam


class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_c=3, num_classes=6,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, qkv_bias=True,
                 qk_scale=None, representation_size=None, distilled=False, drop_ratio=0., mlp_head=True,
                 attn_drop_ratio=0., drop_path_ratio=0., embed_layer=PatchEmbed, norm_layer=None,
                 act_layer=None):

        super(VisionTransformer, self).__init__()
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim  # num_features for consistency with other models
        self.num_tokens = 2 if distilled else 1
        norm_layer = norm_layer or partial(nn.LayerNorm, eps=1e-6)
        act_layer = act_layer or nn.GELU

        self.patch_embed = embed_layer(img_size=img_size, patch_size=patch_size, in_c=in_c, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if distilled else None
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + self.num_tokens, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_ratio)
        self.mlp_head = mlp_head
        dpr = [x.item() for x in torch.linspace(0, drop_path_ratio, depth)]  # stochastic depth decay rule
        self.blocks = nn.Sequential(*[
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                  drop_ratio=drop_ratio, attn_drop_ratio=attn_drop_ratio, drop_path_ratio=dpr[i],
                  norm_layer=norm_layer, act_layer=act_layer)
            for i in range(depth)
        ])
        self.norm = norm_layer(embed_dim)

        # Representation layer
        if representation_size and not distilled:
            self.has_logits = True
            self.num_features = representation_size
            self.pre_logits = nn.Sequential(OrderedDict([
                ("fc", nn.Linear(embed_dim, representation_size)),
                ("act", nn.Tanh())
            ]))
        else:
            self.has_logits = False
            self.pre_logits = nn.Identity()

        # Classifier head(s)
        self.head = nn.Linear(self.num_features, self.num_classes)
        self.head_dist = None
        if distilled:
            self.head_dist = nn.Linear(self.embed_dim, self.num_classes) if num_classes > 0 else nn.Identity()

        # Weight init
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        if self.dist_token is not None:
            nn.init.trunc_normal_(self.dist_token, std=0.02)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(_init_vit_weights)


    def forward_features(self, x):
        # [B, C, H, W] -> [B, num_patches, embed_dim]
        x = self.patch_embed(x)  # [B, 196, 768]
        # [1, 1, 768] -> [B, 1, 768]
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        if self.dist_token is None:
            x = torch.cat((cls_token, x), dim=1)  # [B, 197, 768]
        else:
            x = torch.cat((cls_token, self.dist_token.expand(x.shape[0], -1, -1), x), dim=1)

        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        x = self.norm(x) # cls token + 196 visual tokens ([B, 197, 768])
        if self.mlp_head == False: # default True
                return x[:,1:]
        if self.dist_token is None:
            return self.pre_logits(x[:, 0])
        else:
            return x[:, 0], x[:, 1]

    def forward(self, x):
        x = self.forward_features(x)

        if self.head_dist is not None:
            x, x_dist = self.head(x[0]), self.head_dist(x[1])
            if self.training and not torch.jit.is_scripting():
                # during inference, return the average of both classifier predictions
                return x, x_dist
            else:
                return (x + x_dist) / 2
        else: # head_dist == None
            # default
            if self.mlp_head == False:
                return x
            else:
                # with head
                x = self.head(x)
            return x



def _init_vit_weights(m):

    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=.01)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.zeros_(m.bias)
        nn.init.ones_(m.weight)


def vit_base_patch16_224(num_classes: int = 1000):

    model = VisionTransformer(img_size=224,
                              patch_size=16,
                              embed_dim=768,
                              depth=12,
                              num_heads=12,
                              representation_size=None,
                              num_classes=num_classes)
    return model


def vit_base_patch16_224_in21k(num_classes: int = 21843, has_logits: bool = True):

    model = VisionTransformer(img_size=224,
                              patch_size=16,
                              embed_dim=768,
                              depth=12,
                              num_heads=12,
                              representation_size=768 if has_logits else None,
                              num_classes=num_classes)
    return model


def vit_base_patch32_224(num_classes: int = 1000):

    model = VisionTransformer(img_size=224,
                              patch_size=32,
                              embed_dim=768,
                              depth=12,
                              num_heads=12,
                              representation_size=None,
                              num_classes=num_classes)
    return model


def vit_base_patch32_224_in21k(num_classes: int = 21843, has_logits: bool = True):

    model = VisionTransformer(img_size=224,
                              patch_size=32,
                              embed_dim=768,
                              depth=12,
                              num_heads=12,
                              representation_size=768 if has_logits else None,
                              num_classes=num_classes)
    return model


def vit_large_patch16_224(num_classes: int = 1000):

    model = VisionTransformer(img_size=224,
                              patch_size=16,
                              embed_dim=1024,
                              depth=24,
                              num_heads=16,
                              representation_size=None,
                              num_classes=num_classes)
    return model


def vit_large_patch16_224_in21k(num_classes: int = 21843, has_logits: bool = True):

    model = VisionTransformer(img_size=224,
                              patch_size=16,
                              embed_dim=1024,
                              depth=24,
                              num_heads=16,
                              representation_size=1024 if has_logits else None,
                              num_classes=num_classes)
    return model


def vit_large_patch32_224_in21k(num_classes: int = 21843, has_logits: bool = True):

    model = VisionTransformer(img_size=224,
                              patch_size=32,
                              embed_dim=1024,
                              depth=24,
                              num_heads=16,
                              representation_size=1024 if has_logits else None,
                              num_classes=num_classes)
    return model


def vit_huge_patch14_224_in21k(num_classes: int = 21843, has_logits: bool = True):

    model = VisionTransformer(img_size=224,
                              patch_size=14,
                              embed_dim=1280,
                              depth=32,
                              num_heads=16,
                              representation_size=1280 if has_logits else None,
                              num_classes=num_classes)
    return model
""")

# **model.py**

In [ ]:
with open('model.py', 'w') as f:
    f.write("""
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
import os
import math
import torch
import torch.nn.functional as F
from einops import rearrange, repeat
from torch import nn
from timm.models.layers.weight_init import trunc_normal_
from ViT_model import VisionTransformer
from utils import compute_rotation_matrix_from_ortho6d
import matplotlib.pyplot as plt
import seaborn as sns
MIN_NUM_PATCHES = 16
BN_MOMENTUM = 0.1

class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x


class PreNorm(nn.Module):
    def __init__(self, dim, fn, fusion_factor=1):
        super().__init__()
        self.norm = nn.LayerNorm(dim * fusion_factor)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0., num_ori_tokens=None, scale_with_head=False, show_attns=False,n_dep=0):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5 if scale_with_head else dim ** -0.5

        self.to_qkv = nn.Linear(dim, dim * 3, bias=False)
        self.to_out = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Dropout(dropout)
        )
        self.show_attns = show_attns
        self.num_ori_tokens = num_ori_tokens
        self.n_dep = n_dep

    def plot_attention(self, attn, type="single head", head_index=5):
        if not os.path.exists('output/vis'):
            os.makedirs('output/vis')
        if type == "single head":
            values = attn[0,head_index,0:self.num_ori_tokens,0:self.num_ori_tokens].detach().cpu()
        else: # all heads
            values = torch.sum(attn,dim=1)
            values = values[0, 0:self.num_ori_tokens, 0:self.num_ori_tokens].detach().cpu()

        fig = plt.figure()
        sns.heatmap(values, cmap='plasma')
        fig.savefig(f"./output/vis/attention interaction in layer {self.n_dep+1}.png", bbox_inches='tight')
        plt.show()

    def forward(self, x, mask=None):
        b, n, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)


        # ======================================================================
        # 토큰들이 사진과 정보 공유
        # attn : 9개의 자세 토큰이 196(14*14)개의 시각 토큰 중 어디에 큰 가중치를
        #        두고 있는지 0 ~ 1 사이의 확률값으로 기록
        dots = torch.einsum('bhid,bhjd->bhij', q, k) * self.scale
        mask_value = -torch.finfo(dots.dtype).max
        # ======================================================================

        if mask is not None:
            mask = F.pad(mask.flatten(1), (1, 0), value=True)
            assert mask.shape[-1] == dots.shape[-1], 'mask has incorrect dimensions'
            mask = mask[:, None, :] * mask[:, :, None]
            dots.masked_fill_(~mask, mask_value)
            del mask

        attn = dots.softmax(dim=-1)

        if self.show_attns == True:
            self.plot_attention(attn)

        out = torch.einsum('bhij,bhjd->bhid', attn, v)

        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.to_out(out)
        return out


class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, mlp_dim, dropout, num_ori_tokens=None,
                 all_attn=False, scale_with_head=False, show_attns=False):
        super().__init__()
        self.layers = nn.ModuleList([])
        self.all_attn = all_attn
        self.num_ori_tokens = num_ori_tokens
        for n_dep in range(depth):
            self.layers.append(nn.ModuleList([
                Residual(PreNorm(dim, Attention(dim, heads=heads, dropout=dropout, num_ori_tokens=num_ori_tokens,
                                                scale_with_head=scale_with_head,show_attns=show_attns,n_dep=n_dep))),
                Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout=dropout)))
            ]))

    def forward(self, x, mask=None, pos=None):
        for idx, (attn, ff) in enumerate(self.layers):
            if idx > 0 and self.all_attn:
                x[:, self.num_ori_tokens:] += pos
            x = attn(x, mask=mask)
            x = ff(x)
        return x


class Orientation_Blocks(nn.Module):

    def __init__(self, *, num_ori_tokens, dim, depth, heads, mlp_dim,
                 dropout=0., emb_dropout=0., pos_embedding_type="learnable",
                 ViT_feature_dim, ViT_feature_num, w, h, inference_view=False):

        super().__init__()
        patch_dim = ViT_feature_dim
        self.inplanes = 64
        self.num_ori_tokens = num_ori_tokens
        self.num_patches = ViT_feature_num
        self.pos_embedding_type = pos_embedding_type
        self.all_attn = (self.pos_embedding_type == "sine-full")

        # ======================================================================
        # 자세 토큰 만들어지는 곳
        self.ori_tokens = nn.Parameter(torch.zeros(1, self.num_ori_tokens, dim))
        # ======================================================================

        self._make_position_embedding(w, h, dim, pos_embedding_type)

        self.patch_to_embedding = nn.Linear(patch_dim, dim)
        self.dropout = nn.Dropout(emb_dropout)
        self.inference_view = inference_view

        self.transformer = Transformer(dim, depth, heads, mlp_dim, dropout, num_ori_tokens=num_ori_tokens,
                                       all_attn=self.all_attn, show_attns=self.inference_view)

        self.to_ori_token = nn.Identity()

    def _make_position_embedding(self, w, h, d_model, pe_type='sine'):

        assert pe_type in ['none', 'learnable', 'sine', 'sine-full']
        if pe_type == 'none':
            self.pos_embedding = None
        else:
            with torch.no_grad():
                self.pe_h = h
                self.pe_w = w
                length = self.pe_h * self.pe_w
            if pe_type == 'learnable':
                self.pos_embedding = nn.Parameter(torch.zeros(1, self.num_patches + self.num_ori_tokens, d_model))
                trunc_normal_(self.pos_embedding, std=.02)
            else:
                self.pos_embedding = nn.Parameter(
                    self._make_sine_position_embedding(d_model),
                    requires_grad=False)

    def _make_sine_position_embedding(self, d_model, temperature=10000,
                                      scale=2 * math.pi):
        h, w = self.pe_h, self.pe_w
        area = torch.ones(1, h, w)  # [b, h, w]
        y_embed = area.cumsum(1, dtype=torch.float32)
        x_embed = area.cumsum(2, dtype=torch.float32)

        one_direction_feats = d_model // 2

        eps = 1e-6
        y_embed = y_embed / (y_embed[:, -1:, :] + eps) * scale
        x_embed = x_embed / (x_embed[:, :, -1:] + eps) * scale

        dim_t = torch.arange(one_direction_feats, dtype=torch.float32)
        dim_t = temperature ** (2 * (dim_t // 2) / one_direction_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack(
            (pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack(
            (pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        pos = pos.flatten(2).permute(0, 2, 1)
        return pos

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion, momentum=BN_MOMENTUM),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def plot_sim_matrix(self, A, type="cos-similarity"):

        assert type == "softmax" or type == "cos-similarity", "please use correct type. (softmax/cos-similarity)"

        if not os.path.exists('output/vis'):
            os.makedirs('output/vis')

        if type=="softmax":
            in_pro = A.mm(A.T) / math.sqrt(A.shape[1])
            prob = F.softmax(in_pro, dim=0)
            fig = plt.figure()
            sns.heatmap(prob, cmap='plasma')
            fig.savefig("./output/vis/softmax_similarity_matrix_of_ori_tokens.png", bbox_inches='tight')
            plt.show()
        elif type=="cos-similarity":
            a = (A / torch.norm(A, dim=-1, keepdim=True) )[0,...]
            similarity = torch.mm(a, a.T)
            fig = plt.figure()
            sns.heatmap(similarity, cmap='plasma') # options: YlGnBu plasma
            fig.savefig("./output/vis/cos_similarity_matrix_of_ori_tokens.png", bbox_inches='tight')
            plt.show()

    def forward(self, features, mask=None):

        # show ori_tokens similarity matrix
        if self.inference_view == True:
            self.plot_sim_matrix(self.ori_tokens.cpu())

        # transformer features
        x = self.patch_to_embedding(features) # shape [batch_size, channel=197, dim=192]
        b, n, _ = x.shape

        # add learnable orientation tokens
        ori_tokens = repeat(self.ori_tokens, '() n d -> b n d', b=b)

        # add pos_embedding
        if self.pos_embedding_type in ["sine", "sine-full"]:
            x += self.pos_embedding[:, :n]
            x = torch.cat((ori_tokens, x), dim=1)
        elif self.pos_embedding_type == "learnable":
            x = torch.cat((ori_tokens, x), dim=1)
            x += self.pos_embedding[:, :(n + self.num_ori_tokens)]

        x = self.dropout(x)

        # feature extractor (ViT) -> Orientation_Blocks
        x = self.transformer(x, mask, self.pos_embedding)

        # [수정된 부분] 128차원 원본 토큰 205개 전체 반환
        return x


class TokenHPE(nn.Module):
    # [수정된 부분] 매개변수 image_size, patch_size 추가
    def __init__(self, image_size=224, patch_size=16, num_ori_tokens=9,
                 depth=12, heads=12, embedding='sine-full', ViT_weights='',
                 dim=128, mlp_ratio=3, inference_view=False
                 ):
        super(TokenHPE, self).__init__()

        # Feature extractor (ViT) - 기존 동일
        self.feature_extractor = VisionTransformer(
                              img_size=224,
                              patch_size=16,
                              embed_dim=768,
                              depth=12,
                              num_heads=12,
                              representation_size=None,
                              mlp_head=False,
                              )

        # pretrained 가중치 불러오는 부분 - 기존 동일
        if ViT_weights != "":
            assert os.path.exists(ViT_weights), "weights file: '{}' not exist.".format(ViT_weights)
            weights_dict = torch.load(ViT_weights, map_location="cuda")
            for k in list(weights_dict.keys()):
                if "head" in k:
                    del weights_dict[k]
            print("use pretrained feature extractor (ViT) weights")
            print(self.feature_extractor.load_state_dict(weights_dict, strict=False))

        # Orientation blocks - 기존 동일
        self.Ori_blocks = Orientation_Blocks(
                                         num_ori_tokens=num_ori_tokens,
                                         dim=dim,
                                         ViT_feature_dim=768,
                                         ViT_feature_num=197,
                                         w=14,
                                         h=14,
                                         depth=depth,
                                         heads=heads,
                                         mlp_dim=dim * mlp_ratio,
                                         pos_embedding_type=embedding,
                                         inference_view=inference_view
                                         )

        # [수정된 부분] 전체 토큰 개수(205개) 산출
        fm_tokens = (image_size // patch_size) ** 2
        total_tokens = num_ori_tokens + fm_tokens

        # [수정된 부분] 128차원을 16차원으로 축소하는 Linear 계층 할당
        self.proj_dim = 16
        self.projection = nn.Linear(dim, self.proj_dim)

        # [수정된 부분] 은닉층 차원 할당
        hidden_dim = 256

        # [수정된 부분] 파라미터 최적화 구조 MLP Head 할당
        self.mlp_head = nn.Sequential(
            nn.Linear(total_tokens * self.proj_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # feature extractor (ViT)
        x = self.feature_extractor(x)

        # [수정된 부분] 205개 토큰 전체 추출. 형태: [batch, 205, 128]
        all_tokens = self.Ori_blocks(x)

        # [수정된 부분] 토큰 차원 16으로 축소. 형태: [batch, 205, 16]
        projected_tokens = self.projection(all_tokens)

        # [수정된 부분] 1차원 배열로 평탄화. 형태: [batch, 3280]
        flat_tokens = rearrange(projected_tokens, 'batch tokens dim -> batch (tokens dim)')

        # [수정된 부분] 최종 각도 예측. 형태: [batch, 1]
        pred = self.mlp_head(flat_tokens)

        # [수정된 부분] 9개의 원본 자세 토큰 분리 및 반환 설정
        dir_tokens = all_tokens[:, :9, :]

        return pred, dir_tokens

""")

# **loss.py**

In [ ]:

with open('loss.py', 'w') as f:
    f.write("""
import torch
import torch.nn as nn

# ==========================================================
# 거북목(CVA) 예측 전용 Loss 함수 (초간단 MSE Loss)
# ==========================================================
class CVALoss(nn.Module):
    def __init__(self):
        super(CVALoss, self).__init__()
        # AI의 예측값과 실제 정답의 차이를 제곱해서 평균 내는 함수
        self.mse_loss = nn.MSELoss()

    def forward(self, pred, target):

        # 예측값과 정답 사이의 오차(MSE) 계산
        loss = self.mse_loss(pred, target)

        # 원래 TokenHPE의 train.py 구조(overall_loss, pred_loss, ori_loss 3개 반환)가
        # 에러 나지 않도록, 형식을 맞춰서 리턴해 줍니다.
        # (우리에게는 loss 하나만 중요하므로 나머지는 똑같은 값을 넣어줍니다.)
        return loss, loss, loss
""")

# **datasets.py**

In [ ]:
with open('custom_datasets.py', 'w', encoding='utf-8') as f:
    f.write("""
import os
import json
import torch
from torch.utils.data.dataset import Dataset
from PIL import Image
import torchvision.transforms as T

class TurtleNeckDataset(Dataset):
    def __init__(self, data_dir, json_path, transform=None, image_mode='RGB', train_mode=True):
        self.data_dir = data_dir
        self.image_mode = image_mode
        self.train_mode = train_mode

        # 1. 마스터 정답지 로드 및 필터링
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        self.data = []
        missing_count = 0 # 못 찾은 파일 개수 세기

        for item in raw_data:
            img_name_in_json = item.get('front', item.get('front_image', ''))
            if not img_name_in_json or "없음" in img_name_in_json:
                continue

            img_path = os.path.join(self.data_dir, os.path.basename(img_name_in_json))

            if os.path.exists(img_path):
                self.data.append(item)
            else:
                missing_count += 1
                if missing_count <= 5: # 너무 많이 출력되면 지저분하니까 5개만 샘플로 출력
                    print(f"[경고] 파일을 찾을 수 없음: {img_path}")

        if missing_count > 0:
            print(f"총 {missing_count}개의 이미지를 폴더에서 찾지 못해 제외되었습니다. 경로 확인 필요")

        self.length = len(self.data)

        # 2. [강력 처방 1] 데이터 증강 (Data Augmentation)
        if train_mode:
            self.transform = T.Compose([
                T.Resize((224, 224)),
                T.RandomHorizontalFlip(p=0.5),      # 50% 확률로 좌우 반전
                T.RandomRotation(degrees=10),       # ±10도 회전
                T.ColorJitter(brightness=0.2),      # 밝기 조절
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform if transform else T.Compose([
                T.Resize((224, 224)),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

        print(f"[필터링 완료] 유효 데이터: {self.length}개 / 증강 및 스케일링 활성화")

    def __getitem__(self, index):
        item = self.data[index]
        img_name = os.path.basename(item.get('front', item.get('front_image')))
        img_path = os.path.join(self.data_dir, img_name)

        img = Image.open(img_path).convert(self.image_mode)
        if self.transform:
            img = self.transform(img)

        #  [강력 처방 2] 라벨 스케일링 (100으로 나누기)
        cva_angle = float(item['cva_angle']) / 100.0
        label = torch.FloatTensor([cva_angle])

        R_dummy = torch.eye(3).float()
        return img, R_dummy, label, img_path

    def __len__(self):
        return self.length

def getDataset(dataset, data_dir, filename_list, transformations, train_mode=True):
    return TurtleNeckDataset(data_dir, filename_list, transformations, train_mode=train_mode)
""")
print("custom_datasets.py 업데이트 완료")

custom_datasets.py 업데이트 완료


# **train.py**

In [ ]:
with open('train.py', 'w') as f:
    f.write("""
import os
import argparse
import torch
from torchvision import transforms
import torch.backends.cudnn as cudnn
import custom_datasets
from loss import CVALoss
from model import TokenHPE

def parse_args():
    parser = argparse.ArgumentParser(description='Train TurtleNeck CVA model.')
    parser.add_argument('--gpu', dest='gpu_id', help='GPU device id to use [0]', default=0, type=int)
    parser.add_argument('--num_epochs', dest='num_epochs', help='Maximum number of training epochs.', default=60, type=int)
    parser.add_argument('--batch_size', dest='batch_size', help='Batch size.', default=32, type=int)
    parser.add_argument('--lr', dest='lr', help='Base learning rate.', default=0.00001, type=float)

    parser.add_argument('--dataset', dest='dataset', help='Dataset type.', default='TurtleNeck', type=str)
    parser.add_argument('--data_dir', dest='data_dir', help='Directory path for images.', default='./TurtleNeck_Images', type=str)
    parser.add_argument('--filename_list', dest='filename_list', help='Path to JSON file.', default='./master_annotations.json', type=str)

    parser.add_argument('--snapshot', dest='snapshot', help='Path of model snapshot', default='', type=str)
    parser.add_argument('--weights', dest='weights', help='Pretrained VIT weights', default='', type=str)
    parser.add_argument('--describe', dest='describe', help='Describe saving directory name.', default='CVA_Predictor', type=str)
    parser.add_argument('--output_string', dest='output_string', help='String appended to output snapshots.', default='TurtleNeck', type=str)
    args = parser.parse_args()
    return args

if __name__ == '__main__':
    args = parse_args()
    cudnn.enabled = True
    num_epochs = args.num_epochs
    batch_size = args.batch_size
    gpu = args.gpu_id

    if not os.path.exists('output/snapshots'):
        os.makedirs('output/snapshots')

    summary_name = '{}_{}_batch_size{}'.format('TokenHPE', args.describe, args.batch_size)

    if not os.path.exists('output/snapshots/{}'.format(summary_name)):
        os.makedirs('output/snapshots/{}'.format(summary_name))

    model = TokenHPE(
                 num_ori_tokens=9,
                 depth=3,
                 heads=8,
                 embedding='sine',
                 ViT_weights=args.weights,
                 dim=128,
                 )

    if not args.snapshot == '':
        saved_state_dict = torch.load(args.snapshot)
        model.load_state_dict(saved_state_dict['model_state_dict'])
        print("Intermediate weights used!")

    model.to("cuda")
    print('Loading data and preprocessing...')

    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])

    transformations = transforms.Compose([
        transforms.Resize(240),
        transforms.RandomCrop(224),
        transforms.ToTensor(),
        normalize
    ])

    pose_dataset = custom_datasets.getDataset(
        args.dataset, args.data_dir, args.filename_list, transformations
    )

    train_loader = torch.utils.data.DataLoader(
        dataset=pose_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=1
    )

    crit = CVALoss().cuda(gpu)

    pg = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(pg, lr=args.lr)

    if not args.snapshot == '':
        optimizer.load_state_dict(saved_state_dict['optimizer_state_dict'])

    milestones = [20, 40]
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=milestones, gamma=0.5)

    print('Starting training.')
    for epoch in range(num_epochs):
        loss_sum = .0
        iter = 0

        for i, (images, _, labels, _) in enumerate(train_loader):
            iter += 1
            images = images.cuda(gpu)
            labels = labels.cuda(gpu)

            pred, dir_tokens = model(images)

            # [여기 수정] crit가 반환하는 값 중 첫 번째(진짜 Loss)만 로딩
            # 만약 튜플(보따리)로 나오면 첫 번째 값을, 하나만 나오면 그대로 사용
            loss_outputs = crit(pred, labels)
            loss = loss_outputs[0] if isinstance(loss_outputs, tuple) else loss_outputs

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()

            if (i+1) % 2 == 0:
                print('Epoch [%d/%d],\t Iteration [%d/%d] \t MSE Loss: %.4f' % (
                    epoch+1, num_epochs, i+1, len(pose_dataset)//batch_size + 1, loss.item()
                ))

        scheduler.step()

        if epoch % 1 == 0 and epoch < num_epochs:
            print('Taking snapshot...',
                  torch.save({
                      'epoch': epoch,
                      'model_state_dict': model.state_dict(),
                      'optimizer_state_dict': optimizer.state_dict(),
                  }, 'output/snapshots/' + summary_name + '/' + args.output_string + '_epoch_' + str(epoch+1) + '.tar')
                  )
"""
)


# **--------------**

# **train 시도**

In [ ]:
!python train.py \
  --data_dir "/content/drive/MyDrive/TurtleNeckData/front" \
  --filename_list "/content/drive/MyDrive/TurtleNeckData/Answer/final_GT.json" \
  --lr 0.0001

  # data_dir : 정면 사진 (Final_DATA -> front에 들어가셔서 정면 이미지 내 드라이브에 다운 -> 다운 받고 저장한 경로)
  # filename_list : final_GT.json 파일 경로 (457쌍)

  # 오래 걸리니까 돌아가는 것만 확인되면 바로 밑에 train / validation / test 넘어가도 됩니다

Traceback (most recent call last):
  File "/content/train.py", line 9, in <module>
    from model import TokenHPE
  File "/content/model.py", line 14, in <module>
    import matplotlib.pyplot as plt
  File "/usr/local/lib/python3.12/dist-packages/matplotlib/__init__.py", line 1296, in <module>
    rcParams['backend'] = os.environ.get('MPLBACKEND')
    ~~~~~~~~^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/matplotlib/__init__.py", line 769, in __setitem__
    cval = self.validate[key](val)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/matplotlib/rcsetup.py", line 273, in validate_backend
    if s is _auto_backend_sentinel or backend_registry.is_valid_backend(s):
                                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/matplotlib/backends/registry.py", line 244, in is_valid_backend
    self._ensure_entry_points_loaded()
  File "/usr/local/lib/python3.12/dist-packages/matplo

# **train / validation / test**

In [ ]:
import json
from sklearn.model_selection import train_test_split

# final_GT.json 파일 경로 (457쌍)
json_path = "/content/drive/MyDrive/TurtleNeckData/Answer/final_GT.json"

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 유효한 데이터 필터링
valid_data = [item for item in data if item.get('front', item.get('front_image', '')) and "없음" not in item.get('front', item.get('front_image', ''))]

# Train 70%, 나머지 30% (Val 15%, Test 15%)
train_data, temp_data = train_test_split(valid_data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"총 데이터 수: {len(valid_data)}")
print(f"Train 데이터: {len(train_data)} / Validation 데이터: {len(val_data)} / Test 데이터: {len(test_data)}")

# 분할된 데이터를 각각의 JSON 파일로 저장
with open('train_data.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)
with open('val_data.json', 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=4)
with open('test_data.json', 'w', encoding='utf-8') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=4)

print("데이터 분할 및 저장 완료")

총 데이터 수: 466
Train 데이터: 326 / Validation 데이터: 70 / Test 데이터: 70
데이터 분할 및 저장 완료


In [ ]:
with open('train_with_val.py', 'w') as f:
    f.write("""
import os
import argparse
import torch
import torch.nn.functional as F
from torchvision import transforms
import torch.backends.cudnn as cudnn
import custom_datasets
from loss import CVALoss
from model import TokenHPE

def parse_args():
    parser = argparse.ArgumentParser(description='Train TurtleNeck CVA model with Validation and Test.')
    parser.add_argument('--gpu', dest='gpu_id', default=0, type=int)
    parser.add_argument('--num_epochs', dest='num_epochs', default=60, type=int)
    parser.add_argument('--batch_size', dest='batch_size', default=16, type=int)
    parser.add_argument('--lr', dest='lr', default=0.0001, type=float) # lr 0.0001로 약간 올림
    parser.add_argument('--dataset', dest='dataset', default='TurtleNeck', type=str)
    parser.add_argument('--data_dir', dest='data_dir', default='/content/drive/MyDrive/TurtleNeckData/front', type=str)

    parser.add_argument('--train_json', dest='train_json', default='./train_data.json', type=str)
    parser.add_argument('--val_json', dest='val_json', default='./val_data.json', type=str)
    parser.add_argument('--test_json', dest='test_json', default='./test_data.json', type=str)

    parser.add_argument('--weights', dest='weights', default='', type=str)
    args = parser.parse_args()
    return args

if __name__ == '__main__':
    args = parse_args()
    cudnn.enabled = True
    gpu = args.gpu_id

    model = TokenHPE(num_ori_tokens=9, depth=3, heads=8, embedding='sine', ViT_weights=args.weights, dim=128)

    # ---------------------------------------------------------
    # ViT 뼈대 얼리기 (Freezing) --> 데이터셋이 적을 때 사용
    # 데이터가 적을 때는 뼈대는 학습 안 되게 막아두는 것이 훨씬 좋은 경우
    # 데이터셋이 충분히 확보가 된다면 풀어도 될 듯
    for param in model.feature_extractor.parameters():
        param.requires_grad = False
    # ---------------------------------------------------------

    model.to("cuda")

    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    train_transforms = transforms.Compose([
        transforms.Resize(240),
        transforms.RandomCrop(224),
        transforms.ColorJitter(brightness=0.2),
        transforms.ToTensor(),
        normalize
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize
    ])

    train_dataset = custom_datasets.getDataset(args.dataset, args.data_dir, args.train_json, train_transforms, train_mode=True)
    val_dataset = custom_datasets.getDataset(args.dataset, args.data_dir, args.val_json, val_transforms, train_mode=False)

    train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size=args.batch_size, shuffle=False)

    crit = CVALoss().cuda(gpu)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    print('Starting training with Validation (Loss, MSE & MAE tracking)...')
    for epoch in range(args.num_epochs):
        # --- TRAIN ---
        model.train()
        train_loss_sum = 0.0
        train_mse_sum = 0.0
        train_mae_sum = 0.0

        for i, (images, _, labels, _) in enumerate(train_loader):
            images, labels = images.cuda(gpu), labels.cuda(gpu)
            pred, _ = model(images)

            # 수정: 안전한 Loss 언패킹
            loss_outputs = crit(pred, labels)
            loss = loss_outputs[0] if isinstance(loss_outputs, tuple) else loss_outputs

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 수정: shape 불일치 방지를 위한 squeeze() 적용
            pred_sq = pred.squeeze()
            labels_sq = labels.squeeze()

            # 마지막 배치에서 데이터가 1개만 남았을 때 차원 깨짐 방지
            if pred_sq.dim() == 0:
                pred_sq = pred_sq.unsqueeze(0)
                labels_sq = labels_sq.unsqueeze(0)

            train_loss_sum += loss.item()
            train_mse_sum += F.mse_loss(pred_sq, labels_sq).item()
            train_mae_sum += F.l1_loss(pred_sq, labels_sq).item()

        avg_train_loss = train_loss_sum / len(train_loader)
        avg_train_mse = train_mse_sum / len(train_loader)
        avg_train_mae = train_mae_sum / len(train_loader)

        # --- VALIDATION ---
        model.eval()
        val_loss_sum = 0.0
        val_mse_sum = 0.0
        val_mae_sum = 0.0

        with torch.no_grad():
            for images, _, labels, _ in val_loader:
                images, labels = images.cuda(gpu), labels.cuda(gpu)
                pred, _ = model(images)

                loss_outputs = crit(pred, labels)
                loss = loss_outputs[0] if isinstance(loss_outputs, tuple) else loss_outputs

                pred_sq = pred.squeeze()
                labels_sq = labels.squeeze()

                if pred_sq.dim() == 0:
                    pred_sq = pred_sq.unsqueeze(0)
                    labels_sq = labels_sq.unsqueeze(0)

                val_loss_sum += loss.item()
                val_mse_sum += F.mse_loss(pred_sq, labels_sq).item()
                val_mae_sum += F.l1_loss(pred_sq, labels_sq).item()

        avg_val_loss = val_loss_sum / len(val_loader)
        avg_val_mse = val_mse_sum / len(val_loader)
        avg_val_mae = val_mae_sum / len(val_loader)

        print(f"Epoch [{epoch+1:02d}/{args.num_epochs:02d}] "
              f"| Train - Loss: {avg_train_loss:.4f}, MSE: {avg_train_mse:.4f}, MAE: {avg_train_mae:.4f} "
              f"| Val - Loss: {avg_val_loss:.4f}, MSE: {avg_val_mse:.4f}, MAE: {avg_val_mae:.4f}")

    # --- TEST ---
    print('Starting Testing on Test Dataset...')
    test_dataset = custom_datasets.getDataset(args.dataset, args.data_dir, args.test_json, val_transforms, train_mode=False)
    test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=1, shuffle=False)

    model.eval()
    test_loss_sum = 0.0
    test_mse_sum = 0.0
    test_mae_sum = 0.0

    with torch.no_grad():
        for idx, (images, _, labels, img_paths) in enumerate(test_loader):
            images, labels = images.cuda(gpu), labels.cuda(gpu)
            pred, _ = model(images)

            loss_outputs = crit(pred, labels)
            loss = loss_outputs[0] if isinstance(loss_outputs, tuple) else loss_outputs

            pred_sq = pred.squeeze()
            labels_sq = labels.squeeze()

            if pred_sq.dim() == 0:
                pred_sq = pred_sq.unsqueeze(0)
                labels_sq = labels_sq.unsqueeze(0)

            test_loss_sum += loss.item()
            test_mse_sum += F.mse_loss(pred_sq, labels_sq).item()
            test_mae_sum += F.l1_loss(pred_sq, labels_sq).item()

            pred_angle = pred.item() * 100.0
            true_angle = labels.item() * 100.0

            img_name = os.path.basename(img_paths[0])

            print(f"Test [{idx+1:02d}/{len(test_loader):02d}] Image: {img_name} | True CVA: {true_angle:.2f}° | Pred CVA: {pred_angle:.2f}°")

    avg_test_loss = test_loss_sum / len(test_loader)
    avg_test_mse = test_mse_sum / len(test_loader)
    avg_test_mae = test_mae_sum / len(test_loader)

    print(f"\\n[Final Test Average] Loss: {avg_test_loss:.4f}, MSE: {avg_test_mse:.4f}, MAE: {avg_test_mae:.4f}")

""")
print("train_with_val.py 업데이트 완료! 이제 준비된 Test 데이터 개수에 맞춰 유동적으로 출력.")

train_with_val.py 업데이트 완료! 이제 준비된 Test 데이터 개수에 맞춰 유동적으로 출력.


In [ ]:
# 분할된 데이터로 모델 학습 실행
!python train_with_val.py

[필터링 완료] 유효 데이터: 326개 / 증강 및 스케일링 활성화!
[필터링 완료] 유효 데이터: 70개 / 증강 및 스케일링 활성화!
Starting training with Validation (Loss, MSE & MAE tracking)...
Epoch [01/60] | Train - Loss: 0.0160, MSE: 0.0160, MAE: 0.1007 | Val - Loss: 0.0097, MSE: 0.0097, MAE: 0.0763
Epoch [02/60] | Train - Loss: 0.0134, MSE: 0.0134, MAE: 0.0923 | Val - Loss: 0.0106, MSE: 0.0106, MAE: 0.0798
Epoch [03/60] | Train - Loss: 0.0113, MSE: 0.0113, MAE: 0.0853 | Val - Loss: 0.0076, MSE: 0.0076, MAE: 0.0744
Epoch [04/60] | Train - Loss: 0.0099, MSE: 0.0099, MAE: 0.0795 | Val - Loss: 0.0102, MSE: 0.0102, MAE: 0.0865
Epoch [05/60] | Train - Loss: 0.0105, MSE: 0.0105, MAE: 0.0800 | Val - Loss: 0.0083, MSE: 0.0083, MAE: 0.0714
Epoch [06/60] | Train - Loss: 0.0091, MSE: 0.0091, MAE: 0.0741 | Val - Loss: 0.0048, MSE: 0.0048, MAE: 0.0516
Epoch [07/60] | Train - Loss: 0.0077, MSE: 0.0077, MAE: 0.0684 | Val - Loss: 0.0057, MSE: 0.0057, MAE: 0.0558
Epoch [08/60] | Train - Loss: 0.0072, MSE: 0.0072, MAE: 0.0651 | Val - Loss: 0.0041, MSE:

# **파라미터 체크**

In [ ]:
# 1. 모델 구조 요약을 위한 라이브러리 설치 (코랩 환경)
!pip install torchinfo

# 2. 파라미터 확인 스크립트
import torch
from torchinfo import summary
from model import TokenHPE # 현재 작성자님의 모델 구조 파일

print("모델 구조 및 파라미터 분석 시작")

# 학습할 때와 동일한 세팅으로 모델 인스턴스 생성
model = TokenHPE(num_ori_tokens=9, depth=3, heads=8, embedding='sine', dim=128)

# summary를 통해 모델의 입력부터 출력까지의 흐름과 파라미터 수 계산
# input_size = (배치 사이즈, 채널 수, 이미지 높이, 이미지 너비)
model_summary = summary(model, input_size=(1, 3, 224, 224),
                        col_names=["input_size", "output_size", "num_params", "trainable"],
                        depth=4) # depth를 깊게 주어 세부 레이어까지 확인

print(model_summary)

모델 구조 및 파라미터 분석 시작
Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #                   Trainable
TokenHPE                                                     [1, 3, 224, 224]          [1, 1]                    --                        Partial
├─VisionTransformer: 1-1                                     [1, 3, 224, 224]          [1, 196, 768]             156,678                   True
│    └─PatchEmbed: 2-1                                       [1, 3, 224, 224]          [1, 196, 768]             --                        True
│    │    └─Conv2d: 3-1                                      [1, 3, 224, 224]          [1, 768, 14, 14]          590,592                   True
│    │    └─Identity: 3-2                                    [1, 196, 768]             [1, 196, 768]             --                        --
│    └─Dropout: 2-2                                          [1, 197, 768]             [1, 197, 768]           